In [11]:
using MarineHydro
using Test
using PyCall
using FiniteDifferences
using Zygote
cpt = pyimport("capytaine")

PyObject <module 'capytaine' from 'C:\\Users\\15183\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\capytaine\\__init__.py'>

In [12]:
# Description of problem
omegas = [1.1, 1.5] # frequencies [rad/s]
beta = 0 # incident wave angle [rad]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.7
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)

# Get MarineHydro values
mesh = Mesh(cptmesh)
rigid_dof_list = DOFs
rotation_center = [1.0, 1.0, 0.0] # off set for nonzero off-diagoinal elements
fb = FloatingBody(mesh, rigid_dof_list, rotation_center, "name")


FloatingBody(Mesh([-1.25 -1.5 2.755455298081545e-16; -1.25 -1.0606601717798214 -1.060660171779821; … ; 1.25 1.0606601717798214 -1.0606601717798214; 1.25 1.5 -9.184850993605148e-17], [13 4 9 14; 18 13 14 19; … ; 63 65 66 64; 57 59 58 56], [-1.125 0.5303300858899105 -1.2803300858899107; -0.8749999999999999 0.5303300858899105 -1.2803300858899107; … ; 1.25 0.995812289025486 -0.41247895569215276; 1.2500000000000002 -0.9958122890254862 -0.4124789556921525], [0.0 0.3826834323650895 -0.923879532511287; 0.0 0.3826834323650895 -0.923879532511287; … ; 1.0 0.0 -0.0; 1.0 0.0 -0.0], [0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743  …  0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.5966213466261493, 0.19887378220871652, 0.19887378220871652, 0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.5966213466261494], [0.

In [20]:
fmat = cptmesh.faces
fmat[1,:]

4-element Vector{Int64}:
 13
  4
  9
 14

In [ ]:
# Radiation solve function

function A_and_B_vec(w)
    added_mass_dict, damping_dict = calculate_radiation_forces(fb, w)
    A_vals = [real(added_mass_dict[(w, i, r)]) for i in DOFs, r in DOFs]
    B_vals = [real(damping_dict[(w, i, r)]) for i in DOFs, r in DOFs]
    return vcat(vec(A_vals), vec(B_vals)) # Note this is [A_11,A_12,A_21...,B_22]
end

function Jacobian_of_rad_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        A_and_B_vec(real(w)) 
    end
    return vec(real.(j)[1])
end




Jacobian_of_rad_problem (generic function with 1 method)

In [4]:
FiniteDifferences.central_fdm(5, 1)(A_and_B_vec, 1.04)

8-element Vector{Float64}:
 -2003.4547190538387
 -2003.4547190611584
 -2003.4547190556686
 -1841.683516006058
  4021.271942553311
  4021.271942552076
  4021.271942553357
  4055.533162466031

In [5]:
Jacobian_of_rad_problem(1.04)

([-2003.4547190548653, -2003.4547190548628, -2003.454719054873, -1841.6835160017295, 4021.2719425540427, 4021.2719425540463, 4021.271942554036, 4055.5331624646183],)

In [ ]:
# Incident solve function
function F_FK_vec(w)
    F_FK_dict = FroudeKrylovForce(fb, w)
    F_FK_vals = [F_FK_dict[(w, i)] for i in DOFs]
    return vcat(real.(vec(F_FK_vals)),imag.(vec(F_FK_vals))) # Note this is [real(F_FK_1),real(F_FK_2),...,imag(F_FK_1),imag(F_FK_2),... ]
end

function Jacobian_of_inc_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        F_FK_vec(real(w)) 
    end
    return vec(real.(j)[1])
end

Jacobian_of_inc_problem (generic function with 1 method)

In [16]:
F_FK_vec(1.04)

4-element Vector{Float64}:
 65296.76064364326
 65296.76064364327
    -4.263256414560601e-14
  1043.0019882858892

In [17]:
FiniteDifferences.central_fdm(5, 1)(F_FK_vec, 1.04)

4-element Vector{Float64}:
 -15291.86780301481
 -15291.867802996425
      4.3885805772092515e-11
   2036.6072499605498

In [18]:
Jacobian_of_inc_problem(1.04)

4-element Vector{Float64}:
 -15291.867803001878
 -15291.867803001951
     -1.1728177548788414e-13
   2036.6072499580941

In [19]:
# Diffraction solve function
function F_D_vec(w)
    F_D_dict = DiffractionForce(fb, w)
    F_D_vals = [F_D_dict[(w, i)] for i in DOFs]
    return vcat(real.(vec(F_D_vals)),imag.(vec(F_D_vals))) # Note this is [real(F_D_1),real(F_D_2),...,imag(F_D_1),imag(F_D_2),... ]
end

function Jacobian_of_diff_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        F_D_vec(real(w)) 
    end
    return vec(real.(j)[1])
end

Jacobian_of_diff_problem (generic function with 1 method)

In [20]:
F_D_vec(1.04)

4-element Vector{Float64}:
 -7171.8107224722235
 -7188.246081179137
 -2101.171103820327
   897.3545590831112

In [21]:
FiniteDifferences.central_fdm(5, 1)(F_D_vec, 1.04)

4-element Vector{Float64}:
 -10797.835211840546
 -10920.751417979462
  -6232.610093343011
   -342.18323846108996

In [22]:
Jacobian_of_diff_problem(1.04)

4-element Vector{Float64}:
 -10797.835211842266
 -10920.751417980166
  -6232.610093343079
   -342.18323846119165

In [ ]:
# Make tests
@testset "Gradient accuracy check with Finite diff [w.r.t omega] for MDOF cylnder" begin
    # Description of problem
    omegas = [1.1] # frequencies [rad/s]
    beta = 0 # incident wave angle [rad]
    t_DOFs = ["Heave"] # translational DOFs
    r_DOFs = ["Pitch"] # rotational DOFs
    DOFs = [t_DOFs; r_DOFs] # all DOFs

    # Create Mesh object
    radius = 1.5  
    center = (0.0,0.0,0.0) 
    len = 2.5
    faces_max_radius = 0.7
    cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
                radius=radius,
                center=center, 
                length=len, 
                faces_max_radius = faces_max_radius
                ).keep_immersed_part(inplace=true)

    # Get MarineHydro values
    mesh = Mesh(cptmesh)
    rigid_dof_list = DOFs
    rotation_center = [1.0, 1.0, 0.0] # off set for nonzero off-diagoinal elements
    fb = FloatingBody(mesh, rigid_dof_list, rotation_center)

    # Radiation solve functions
    function A_and_B_vec(w)
        added_mass_dict, damping_dict = calculate_radiation_forces(fb, w)
        A_vals = [real(added_mass_dict[(w, i, r)]) for i in DOFs, r in DOFs]
        B_vals = [real(damping_dict[(w, i, r)]) for i in DOFs, r in DOFs]
        return vcat(vec(A_vals), vec(B_vals)) # Note this is [A_11,A_12,A_21...,B_22]
    end
    function Jacobian_of_rad_problem(Omega)
        # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
        j = Zygote.jacobian(Omega + 0im) do w
            A_and_B_vec(real(w)) 
        end
        return vec(real.(j)[1])
    end

    # Incident solve function
    function F_FK_vec(w)
        F_FK_dict = FroudeKrylovForce(fb, w)
        F_FK_vals = [F_FK_dict[(w, i)] for i in DOFs]
        return vcat(real.(vec(F_FK_vals)),imag.(vec(F_FK_vals))) # Note this is [real(F_FK_1),real(F_FK_2),...,imag(F_FK_1),imag(F_FK_2),... ]
    end
    function Jacobian_of_inc_problem(Omega)
        # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
        j = Zygote.jacobian(Omega + 0im) do w
            F_FK_vec(real(w)) 
        end
        return vec(real.(j)[1])
    end

    # Diffraction solve function
    function F_D_vec(w)
        F_D_dict = DiffractionForce(fb, w)
        F_D_vals = [F_D_dict[(w, i)] for i in DOFs]
        return vcat(real.(vec(F_D_vals)),imag.(vec(F_D_vals))) # Note this is [real(F_D_1),real(F_D_2),...,imag(F_D_1),imag(F_D_2),... ]
    end
    function Jacobian_of_diff_problem(Omega)
        # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
        j = Zygote.jacobian(Omega + 0im) do w
            F_D_vec(real(w)) 
        end
        return vec(real.(j)[1])
    end

    for omega in omegas
        @testset "Verify sensitivity of added mass and damping wrt Omega: $omega" begin
            A_and_B_FD = FiniteDifferences.central_fdm(5, 1)(A_and_B_vec, omega)
            A_and_B_AD = Jacobian_of_rad_problem(omega)
            @test A_and_B_AD !== nothing
            @test typeof(A_and_B_AD) == Vector{Float64}
            @test A_and_B_AD ≈ A_and_B_FD atol=1e-6 rtol=1e-6
        end
        @testset "Verify sensitivity of Froude Krylov force wrt Omega: $omega" begin
            F_FK_FD = FiniteDifferences.central_fdm(5, 1)(F_FK_vec, omega)
            F_FK_AD = Jacobian_of_inc_problem(omega)
            @test F_FK_AD !== nothing
            @test typeof(F_FK_AD) == Vector{Float64}
            @test F_FK_AD ≈ F_FK_FD atol=1e-6 rtol=1e-6
        end
        @testset "Verify sensitivity of diffraction force wrt Omega: $omega" begin
            F_D_FD = FiniteDifferences.central_fdm(5, 1)(F_D_vec, omega)
            F_D_AD = Jacobian_of_diff_problem(omega)
            @test F_D_AD !== nothing
            @test typeof(F_D_AD) == Vector{Float64}
            @test F_D_AD ≈ F_D_FD atol=1e-6 rtol=1e-6
        end
    end   

end

Test Summary:                                                           | Pass  Total     Time
Gradient accuracy check with Finite diff [w.r.t omega] for MDOF cylnder |    9      9  2m57.6s


Test.DefaultTestSet("Gradient accuracy check with Finite diff [w.r.t omega] for MDOF cylnder", Any[Test.DefaultTestSet("Verify sensitivity of added mass and damping wrt Omega: 1.1", Any[], 3, false, false, true, 1.772124891063e9, 1.772125002652e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl"), Test.DefaultTestSet("Verify sensitivity of Froude Krylov force wrt Omega: 1.1", Any[], 3, false, false, true, 1.772125002652e9, 1.772125003601e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl"), Test.DefaultTestSet("Verify sensitivity of diffraction force wrt Omega: 1.1", Any[], 3, false, false, true, 1.772125003601e9, 1.772125068687e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl")], 0, false, false, true, 1.772124891055e9, 1.772125068687e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_note

In [35]:
a = 0.25:0.25:2
length(a)

8